# PDDL Attack Path Evaluation
Evaluate generated PDDL attack paths: solvability, syntax, and semantic quality.

## 1. Environment Setup (imports, constnats, global variables)

In [8]:
import sys
import os
import re
import json
import time
import random as _rng
try:
    import resource  # Unix-only
except ImportError:
    resource = None
import platform
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from openai import OpenAI
from jinja2 import Environment, FileSystemLoader
from ipywidgets import interact, IntSlider

from sklearn.metrics import classification_report, precision_recall_curve
from sklearn.model_selection import GroupKFold
from statsmodels.stats.inter_rater import fleiss_kappa as _fleiss_kappa, aggregate_raters

from sentence_transformers import SentenceTransformer, util as st_util
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline as hf_pipeline

from cve2pddlap.core.data_loader import load_few_shot_pool
from cve2pddlap.evaluation import create_ff_checker, create_enhsp_checker
from cve2pddlap.evaluation.problem_pddl_generator import generate_problem
from cve2pddlap.llm_providers.remote.openai_compat import QwenProvider

# Project root (relative — works from notebooks/attack_paths/)
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

# Data paths and file names
DATASET_PATH = os.path.join(PROJECT_ROOT, 'resources', 'data', 'CVE-PDDL-NNL-ReAP')
BAD_PDDL_DIR = os.path.join(PROJECT_ROOT, 'resources', 'data', 'mutated_bad_PDDL_AP')
TARGET_POOL_FILE = os.path.join(PROJECT_ROOT, 'resources', 'data', 'target_pool.json')
GENERATED_DOMAIN_DIR = os.path.join(PROJECT_ROOT, 'generated_domain')
FIG_DIR = os.path.join(PROJECT_ROOT, 'Fig', 'embedding_intrinsic')
os.makedirs(FIG_DIR, exist_ok=True)
FIG_DIR_EXT = os.path.join(PROJECT_ROOT, 'Fig', 'embedding_extrinsic')
os.makedirs(FIG_DIR_EXT, exist_ok=True)
EVAL_SET_DIR = os.path.join(GENERATED_DOMAIN_DIR, 'eval_set')
PROMPTS_PATH = os.path.join(PROJECT_ROOT, 'resources', 'prompt', 'evaluation')
AP_PATTERN = re.compile(r'^AP\d+$')
DOMAIN_FILE = 'domain.pddl'
PROBLEM_FILE = 'problem.pddl'

# Models (small defaults — replace with preferred models)
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
# EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-8B"  # too large for CPU
# EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-4B"  # Qwen3 embedding model 4B
EMBEDDING_MODEL_NAME_2 = "BAAI/bge-base-en-v1.5"  # fallback: smaller model


LLM_MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

# Generation parameters
SEED = 42
TEMPERATURE = 0.0
TOP_K = 1


@dataclass(frozen=True)
class EvaluationFlags:
    syntax_check: bool = True
    embedding_intrinsic: bool = True
    embedding_extrinsic: bool = True
    llm_intrinsic: bool = True
    llm_extrinsic: bool = True


eval_flags = EvaluationFlags()

os.environ['MallocStackLogging'] = '0'

# Results output directory
RESULTS_BASE = os.path.join(PROJECT_ROOT, "results", "tests", "reference_set")

def get_device_info():
    """Return device info dict."""
    return {
        "platform": platform.platform(),
        "processor": platform.processor(),
        "python": platform.python_version(),
        "torch_device": "cuda" if torch.cuda.is_available() else "cpu",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A",
    }


## 2. Load data, Prompts, LLM (tokenizer and embedding model)

In [9]:
few_shot_pool = load_few_shot_pool(DATASET_PATH)
print(f'Reference examples: {len(few_shot_pool)}')
for ex in few_shot_pool[:5]:
    print(f'  {ex.key}')
print('  ...')
print(f'Generated domain dir to be evaluated: {os.path.relpath(EVAL_SET_DIR)}')

Reference examples: 55
  CVE-2022-1471 / AP1
  CVE-2022-40149 / AP1
  CVE-2022-40149 / AP2
  CVE-2022-40150 / AP1
  CVE-2022-40150 / AP2
  ...
Generated domain dir to be evaluated: ../../generated_domain/eval_set


In [10]:
def model_short_name(model_name):
    """Generate a short readable name from a full model identifier.
    e.g. 'meta/llama-3.3-70b-instruct' -> 'llama-3.3-70b'
         'gpt-4.1-mini' -> 'gpt-4.1-mini'
         'all-MiniLM-L6-v2' -> 'all-MiniLM-L6-v2'
         'BAAI/bge-base-en-v1.5' -> 'bge-base-en-v1.5'
         'Qwen/Qwen3-Embedding-4B' -> 'Qwen3-Embedding-4B'
    """
    name = model_name.split("/")[-1]  # strip org prefix
    for suffix in ["-instruct", "-Instruct", "-chat", "-Chat"]:
        if name.endswith(suffix):
            name = name[:-len(suffix)]
            break
    return name


def load_target_pool(target_pool_file):
    """Load CVE descriptions from target_pool.json. Returns {cve_id: description}."""
    with open(target_pool_file, encoding='utf-8') as f:
        pool = json.load(f)
    return {entry['cve_id']: entry['description'] for entry in pool}


def load_dataset(data_path, cve_descriptions):
    """Load all CVEs with their descriptions (from target_pool) and attack path PDDL files."""
    dataset = []
    for cve_dir in sorted(Path(data_path).iterdir()):
        if not cve_dir.is_dir():
            continue
        cve_id = cve_dir.name
        description = cve_descriptions.get(cve_id)
        if description is None:
            continue
        attack_paths = []
        for ap_dir in sorted(cve_dir.iterdir()):
            if not ap_dir.is_dir() or not AP_PATTERN.match(ap_dir.name):
                continue
            domain_file = ap_dir / DOMAIN_FILE
            problem_file = ap_dir / PROBLEM_FILE
            if domain_file.exists() and problem_file.exists():
                attack_paths.append({
                    'ap_id': ap_dir.name,
                    'domain': domain_file.read_text(encoding='utf-8').strip(),
                    'problem': problem_file.read_text(encoding='utf-8').strip(),
                })
        dataset.append({
            'cve_id': cve_id,
            'description': description,
            'attack_paths': attack_paths,
        })
    return dataset


def load_bad_dataset(bad_pddl_dir, cve_descriptions):
    """Load mutated bad PDDL domains from mutated_bad_PDDL_AP/.
    Returns list of {cve_id, description, attack_paths: [{ap_id, domain}]}."""
    bad_dataset = []
    bad_dir = Path(bad_pddl_dir)
    if not bad_dir.exists():
        print(f"WARNING: {bad_pddl_dir} not found")
        return bad_dataset
    for cve_dir in sorted(bad_dir.iterdir()):
        if not cve_dir.is_dir():
            continue
        cve_id = cve_dir.name
        description = cve_descriptions.get(cve_id, "")
        attack_paths = []
        for ap_dir in sorted(cve_dir.iterdir()):
            if not ap_dir.is_dir():
                continue
            domain_file = ap_dir / DOMAIN_FILE
            if domain_file.exists():
                attack_paths.append({
                    'ap_id': ap_dir.name,
                    'domain': domain_file.read_text(encoding='utf-8').strip(),
                })
        if attack_paths:
            bad_dataset.append({
                'cve_id': cve_id,
                'description': description,
                'attack_paths': attack_paths,
            })
    return bad_dataset

def load_prompts(prompts_path):
    """Load Jinja2 evaluation prompt templates."""
    return Environment(loader=FileSystemLoader(prompts_path))


def load_embedding_model(model_name):
    """Load a SentenceTransformer bi-encoder model.
    GPU for small models (<2B params), CPU for large models.
    """
    TRUST_REMOTE = ["Qwen3-Embedding", "nomic-ai/", "jinaai/jina-embeddings-v3"]
    # Models known to be too large for GPU encode (>2GB weights)
    FORCE_CPU = ["Qwen3-Embedding-4B", "Qwen3-Embedding-8B", "jina-embeddings-v3", "stella_en_1.5B"]
    kwargs = {"trust_remote_code": True} if any(t in model_name for t in TRUST_REMOTE) else {}
    if any(t in model_name for t in FORCE_CPU):
        model = SentenceTransformer(model_name, device="cpu", **kwargs)
        print(f"  Loaded {model_name} on CPU (large model)")
        return model
    try:
        model = SentenceTransformer(model_name, **kwargs)
        print(f"  Loaded {model_name} on {model.device}")
        return model
    except (RuntimeError, torch.cuda.OutOfMemoryError):
        import gc; gc.collect(); torch.cuda.empty_cache()
        model = SentenceTransformer(model_name, device="cpu", **kwargs)
        print(f"  GPU OOM, loaded {model_name} on CPU")
        return model

def load_llm(model_name):
    """Load a HuggingFace causal LLM with its tokenizer."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map='auto',
    )
    gen = hf_pipeline(
        'text-generation',
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
        do_sample=False,
    )
    return gen, tokenizer



def strip_base_score(cvss_nl):
    """Remove Base Score from CVSS vector NL string."""
    if not cvss_nl:
        return ""
    return _re.sub(r"\s*Base Score:\s*[\d.]+\.?\s*", "", cvss_nl).strip()

def build_nl_ti_selected(entry_ti):
    """Build TI-selected NL: desc + capec + cvss_vector_nl (no base score)."""
    parts = [entry_ti.get("description", "")]
    capec = entry_ti.get("capec", "")
    if capec:
        parts.append(capec)
    cvss = strip_base_score(entry_ti.get("cvss_vector_nl", ""))
    if cvss:
        parts.append(cvss)
    return " ".join(parts)

cve_descriptions = load_target_pool(TARGET_POOL_FILE)
dataset = load_dataset(DATASET_PATH, cve_descriptions)
bad_dataset = load_bad_dataset(BAD_PDDL_DIR, cve_descriptions)
print(f"Bad (mutated) examples: {len(bad_dataset)} CVEs, {sum(len(e['attack_paths']) for e in bad_dataset)} domains")
prompt_env = load_prompts(PROMPTS_PATH)


embedding_model = None
if eval_flags.embedding_intrinsic or eval_flags.embedding_extrinsic:
    embedding_model = load_embedding_model(EMBEDDING_MODEL_NAME)

llm, tokenizer = None, None
if eval_flags.llm_intrinsic or eval_flags.llm_extrinsic:
    llm, tokenizer = load_llm(LLM_MODEL_NAME)

# --- Calibration data for LLM-as-expert evaluation ---
CALIBRATION_DATA_PATH = os.path.join(PROMPTS_PATH, "data.jsonl")

def load_calibration_data(data_jsonl_path, eval_type="intrinsic", exclude_cve=None, n=4, seed=42):
    """Sample n calibration examples from data.jsonl.
    
    Constraints:
        - exclude_cve: the CVE being evaluated is excluded (prevent data leakage)
        - n >= 2: at least 1 good + 1 bad example guaranteed
        - balanced: samples from both calibration_good and calibration_bad pools
    
    Args:
        data_jsonl_path: path to data.jsonl
        eval_type: 'intrinsic' or 'extrinsic'
        exclude_cve: CVE ID to exclude
        n: total number of calibration examples (>= 2)
        seed: random seed for reproducibility
    """
    rng = _rng.Random(seed)
    
    with open(data_jsonl_path) as f:
        all_data = [json.loads(line) for line in f]
    
    pool = [d for d in all_data
            if d.get("eval_type") == eval_type
            and d.get("role", "").startswith("calibration")
            and d.get("cve_id") != exclude_cve]
    
    good = [d for d in pool if d.get("role") == "calibration_good"]
    bad = [d for d in pool if d.get("role") == "calibration_bad"]
    
    n = max(n, 2)  # enforce minimum 2
    n_good = max(1, n // 2)       # at least 1 good
    n_bad = max(1, n - n_good)    # at least 1 bad
    # Adjust if one pool is too small
    n_good = min(n_good, len(good))
    n_bad = min(n_bad, len(bad))
    
    selected_good = rng.sample(good, n_good) if good else []
    selected_bad = rng.sample(bad, n_bad) if bad else []
    
    return selected_good + selected_bad


# --- Rate limit retry wrapper ---
def api_call_with_retry(func, *args, max_retries=5, base_delay=5, **kwargs):
    """Call func with exponential backoff on rate limit errors."""
    for attempt in range(max_retries):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if "429" in str(e) or "rate" in str(e).lower():
                delay = base_delay * (2 ** attempt)
                print(f"  [RATE LIMIT] retry {attempt+1}/{max_retries} in {delay}s...")
                time.sleep(delay)
            else:
                raise
    raise RuntimeError(f"Max retries ({max_retries}) exceeded")


# --- Classification report helpers (following Marco's pattern) ---

def report_row(y_true, y_pred, **meta):
    """Flatten classification_report into a single dict row, with TPR/FPR."""
    rpt = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    row = dict(meta)
    for key, val in rpt.items():
        if isinstance(val, dict):
            for metric, v in val.items():
                row[f'{key}__{metric}'] = v
        else:
            row[key] = val
    row['tpr'] = rpt.get('1', {}).get('recall', float('nan'))
    row['fpr'] = 1.0 - rpt.get('0', {}).get('recall', float('nan'))
    return row

def print_report(y_true, y_pred, title=''):
    if title:
        print(f'\n{title}')
        print('─' * len(title))
    print(classification_report(y_true, y_pred, zero_division=0))


def save_cv_record(cv_path, model_name, threshold, fold_thresholds, row, labels, build_seconds, cv_seconds, cv_method="reference_full_matrix"):
    """Build CV record, dedup by model, and save to JSONL."""
    record = {
        "timestamp": datetime.now().isoformat(),
        "model": model_name,
        "cv_method": cv_method,
        "threshold": threshold,
        "fold_thresholds": fold_thresholds,
        "accuracy": row.get("accuracy", float("nan")),
        "precision": row.get("1__precision", float("nan")),
        "recall": row.get("1__recall", float("nan")),
        "f1": row.get("1__f1-score", float("nan")),
        "tpr": row["tpr"],
        "fpr": row["fpr"],
        "n_positive_pairs": int(labels.sum()),
        "n_negative_pairs": int((labels == 0).sum()),
        "build_pairs_seconds": build_seconds,
        "cv_calibration_seconds": cv_seconds,
    }
    existing = []
    if os.path.exists(cv_path):
        with open(cv_path) as f:
            existing = [json.loads(line) for line in f if line.strip()]
        existing = [r for r in existing if not (r.get("model") == model_name and r.get("cv_method", "reference_full_matrix") == cv_method)]
    existing.append(record)
    with open(cv_path, "w") as f:
        for r in existing:
            f.write(json.dumps(r) + "\n")
    return record


def save_intrinsic_similarity_result(results_path, source, model_name, threshold, cve_id, ap_id, similarity, prediction, elapsed, cv_method="reference_full_matrix"):
    result = {
        "timestamp": datetime.now().isoformat(),
        "source": source,
        "cve_id": cve_id,
        "ap_id": ap_id,
        "model": model_name,
        "cv_method": cv_method,
        "threshold": threshold,
        "similarity": round(similarity, 6),
        "prediction": prediction,
        "elapsed_seconds": round(elapsed, 4),
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_extrinsic_similarity_result(results_path, source, model_name, threshold, cve_id, test_ap, best_ref_ap, similarity, prediction, n_references, elapsed):
    result = {
        "timestamp": datetime.now().isoformat(),
        "source": source,
        "model": model_name,
        "cv_method": cv_method,
        "threshold": threshold,
        "cve_id": cve_id,
        "test_ap": test_ap,
        "best_ref_ap": best_ref_ap,
        "similarity": round(similarity, 6),
        "prediction": prediction,
        "n_references": n_references,
        "elapsed_seconds": round(elapsed, 4),
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_intrinsic_binary_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, label, response, usage, price_in, price_out):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "ap_id": ap_id,
        "label": label, "response_length": len(response),
        "parse_success": label is not None,
        "llm_response": response,
        "usage": {**usage, "cost_usd": round((usage.get("prompt_tokens", 0) * price_in + usage.get("completion_tokens", 0) * price_out) / 1000, 6)},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_intrinsic_scored_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, verdict, final_score, scores, response, usage, price_in, price_out):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "ap_id": ap_id,
        "verdict": verdict, "final_score": final_score, "scores": scores,
        "response_length": len(response),
        "parse_success": "parse_error" not in scores,
        "llm_response": response,
        "usage": {**usage, "cost_usd": round((usage.get("prompt_tokens", 0) * price_in + usage.get("completion_tokens", 0) * price_out) / 1000, 6)},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_extrinsic_binary_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, label, n_references, parse_success, usage, elapsed, cost):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "generated": ap_id,
        "label": label, "n_references": n_references,
        "parse_success": parse_success,
        "usage": {**usage, "elapsed_seconds": elapsed, "cost_usd": cost},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_extrinsic_scored_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, verdict, final_score, n_references, parse_success, usage, elapsed, cost):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "generated": ap_id,
        "verdict": verdict, "final_score": final_score,
        "n_references": n_references,
        "parse_success": parse_success,
        "usage": {**usage, "elapsed_seconds": elapsed, "cost_usd": cost},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def prepare_intrinsic_texts(dataset):
    """Extract texts and build 21x55 pair structure (matches build_intrinsic_pairs).
    Returns:
        descs: 21 unique CVE descriptions (one per CVE)
        domains: 55 reference domains (one per attack path)
        domain_cve_ids: 55 CVE IDs (domain-side, kept for backward compatibility)
        labels: 1155 pair labels (1 if same CVE, else 0), iterated descs-major
        groups: 1155 description-side CVE IDs (for GroupKFold)
    """
    descs = [entry["description"] for entry in dataset]
    desc_cve_ids = [entry["cve_id"] for entry in dataset]
    domains, domain_cve_ids = [], []
    for entry in dataset:
        for ap in entry["attack_paths"]:
            domains.append(ap["domain"])
            domain_cve_ids.append(entry["cve_id"])
    n_d, n_p = len(descs), len(domains)
    labels = np.array([1 if desc_cve_ids[i] == domain_cve_ids[j] else 0
                       for i in range(n_d) for j in range(n_p)])
    groups = np.array([desc_cve_ids[i] for i in range(n_d) for j in range(n_p)])
    return descs, domains, domain_cve_ids, labels, groups


def compute_intrinsic_scores(descs, domains, model):
    """Compute 21x55 similarity scores (flattened, descs-major)."""
    sim_matrix = embedding_similarity_intrinsic(descs, domains, model)
    n_d, n_p = len(descs), len(domains)
    scores = np.array([float(sim_matrix[i, j]) for i in range(n_d) for j in range(n_p)])
    return scores


def prepare_extrinsic_texts(dataset):
    """Extract texts and build pair structure for extrinsic (model-independent)."""
    all_entries = []
    for entry in dataset:
        for ap in entry["attack_paths"]:
            all_entries.append((ap["domain"], entry["cve_id"], ap["ap_id"]))
    domains = [p[0] for p in all_entries]
    cve_ids = [p[1] for p in all_entries]
    n = len(all_entries)
    labels = np.array([1 if cve_ids[i] == cve_ids[j] else 0 for i in range(n) for j in range(i+1, n)])
    groups = np.array([cve_ids[i] for i in range(n) for j in range(i+1, n)])
    return domains, cve_ids, labels, groups


def compute_extrinsic_scores(domains, model):
    """Compute similarity scores for pre-prepared extrinsic texts."""
    E = model.encode(domains, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
    sim_matrix = (E @ E.T).float().cpu().numpy()
    n = len(domains)
    scores = np.array([float(sim_matrix[i, j]) for i in range(n) for j in range(i+1, n)])
    return scores


Bad (mutated) examples: 18 CVEs, 55 domains
  Loaded all-MiniLM-L6-v2 on cpu


Device set to use cpu


## 3. Evaluation

Select two example:
1. reference
2. bad one

In [11]:
# ── Build test samples: ALL reference + ALL bad ──
test_samples = []

# All reference APs
for entry in dataset:
    for ap in entry["attack_paths"]:
        test_samples.append(("reference", entry["cve_id"], ap["ap_id"], ap["domain"]))

# All bad (mutated) APs
for entry in bad_dataset:
    for ap in entry["attack_paths"]:
        test_samples.append(("bad", entry["cve_id"], ap["ap_id"], ap["domain"]))

print(f"Test samples: {len(test_samples)}")
n_ref = sum(1 for s in test_samples if s[0] == "reference")
n_bad = sum(1 for s in test_samples if s[0] == "bad")
print(f"  reference: {n_ref}, bad: {n_bad}")


Test samples: 110
  reference: 55, bad: 55


Metric: domain statistics

In [12]:
def domain_stats(domain_pddl):
    """Extract structural stats from a PDDL domain string."""
    return {
        "domain_size_bytes": len(domain_pddl.encode("utf-8")),
        "n_actions": len(re.findall(r"\(:action\s", domain_pddl)),
        "n_predicates": len(re.findall(r"\([\w-]+", re.findall(r"\(:predicates([^)]*(?:\([^)]*\))*[^)]*?)\)", domain_pddl, re.DOTALL)[0])) if re.findall(r"\(:predicates", domain_pddl) else 0,
        "n_types": len(re.findall(r"\(:types([^)]*?)\)", domain_pddl, re.DOTALL)[0].split()) if re.findall(r"\(:types", domain_pddl) else 0,
    }

Metric: peak memory in MB (gpu/cpu)

In [13]:
def get_peak_rss_mb():
    """Get current process peak RSS in MB (cross-platform).
    Unix: uses resource.getrusage (macOS: bytes, Linux: KB).
    Windows / fallback: uses psutil if available, else returns None."""
    if resource is not None:
        ru = resource.getrusage(resource.RUSAGE_CHILDREN)
        if platform.system() == "Darwin":
            return round(ru.ru_maxrss / 1024 / 1024, 2)
        return round(ru.ru_maxrss / 1024, 2)
    try:
        import psutil
        return round(psutil.Process().memory_info().rss / 1024 / 1024, 2)
    except ImportError:
        return None


### 3.1 Syntax Check (ENHSP)

In [14]:
enhsp = create_enhsp_checker()
# ── Run syntax check ──
t_start_syntax = time.time()
reference_syntax_results = []

save_dir = os.path.join(RESULTS_BASE, "syntax")
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, "results_syntax.jsonl")

open(results_path, "w").close()

for source, cve_id, ap_id, domain_pddl in test_samples:
    try:
        problem_str = generate_problem(domain_pddl)
    except ValueError as e:
        # e.g. remove_stride_goal mutant has no STRIDE goal
        stats = domain_stats(domain_pddl)
        result = {"timestamp": datetime.now().isoformat(),
            "source": source, "cve_id": cve_id, "ap_id": ap_id,
            "syntax_ok": False, "error": str(e), "elapsed_seconds": 0.0, **stats}
        reference_syntax_results.append(result)
        with open(results_path, "a") as f:
            f.write(json.dumps(result) + "\n")
        print(f"[{source}] {cve_id}/{ap_id}  syntax: SKIP ({e})")
        continue
    t0 = time.time()
    r_enhsp = enhsp.check_from_string(domain_pddl, problem_str)
    elapsed = time.time() - t0
    stats = domain_stats(domain_pddl)

    result = {"timestamp": datetime.now().isoformat(),
        "source": source, "cve_id": cve_id, "ap_id": ap_id,
        "syntax_ok": r_enhsp.success, "error": r_enhsp.error,
        "elapsed_seconds": round(elapsed, 4), **stats}
    reference_syntax_results.append(result)
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    print(f"[{source}] {cve_id}/{ap_id}  syntax: {r_enhsp.success}  {elapsed:.3f}s")

t_syntax = time.time() - t_start_syntax
n_pass = sum(1 for r in reference_syntax_results if r["syntax_ok"])
print(f"Syntax: {n_pass}/{len(reference_syntax_results)} passed, time: {t_syntax:.2f}s")


[reference] CVE-2022-1471/AP1  syntax: True  2.462s
[reference] CVE-2022-40149/AP1  syntax: True  30.051s
[reference] CVE-2022-40149/AP2  syntax: True  30.037s
[reference] CVE-2022-40150/AP1  syntax: True  30.038s
[reference] CVE-2022-40150/AP2  syntax: True  30.036s
[reference] CVE-2023-2976/AP1  syntax: True  30.032s
[reference] CVE-2023-2976/AP2  syntax: True  30.034s
[reference] CVE-2023-2976/AP3  syntax: True  30.032s
[reference] CVE-2023-33202/AP1  syntax: True  0.444s
[reference] CVE-2023-33202/AP2  syntax: True  0.348s
[reference] CVE-2023-33202/AP3  syntax: True  0.345s
[reference] CVE-2023-34055/AP1  syntax: True  0.344s
[reference] CVE-2023-44487/AP1  syntax: True  0.360s
[reference] CVE-2023-46589/AP1  syntax: True  30.031s
[reference] CVE-2023-46589/AP2  syntax: True  30.029s
[reference] CVE-2023-46589/AP3  syntax: True  30.031s
[reference] CVE-2023-46589/AP4  syntax: True  30.033s
[reference] CVE-2023-6378/AP1  syntax: True  0.403s
[reference] CVE-2024-12798/AP1  syntax: 